# makemore_backprop — becoming a backprop ninja

Lecture 5: manual backpropagation drills. The forward pass is split into small chunks; write the backward pass by hand and check each gradient against PyTorch autograd.

4 exercises:
- **Ex 1**: manual backprop through the entire MLP, step by step (the heavy one)
- **Ex 2**: cross-entropy loss in one shot
- **Ex 3**: BatchNorm in one shot
- **Ex 4**: full training with your own backward pass

The boilerplate (top of this notebook) is carried over from lec04, with the model definition adjusted to match the official notebook.

In [ ]:
# import module
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# read file
words = open('../names.txt', 'r').read().splitlines()

In [ ]:
# utility functions
from string import ascii_lowercase
stoi = {c:i+1 for i, c in enumerate(ascii_lowercase)}
stoi['.'] = 0
itos = {i:c for c, i in stoi.items()}
vocab_size = len(itos)

# New in this lecture (copy from Karpathy): 
# utility for comparing manual gradients against PyTorch's
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
# build and split the dataset
block_size = 3

def build_dataset(words):  
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])        # 80% training set
Xdev, Ydev = build_dataset(words[n1:n2])    # 10% dev set
Xte, Yte = build_dataset(words[n2:])        # 10% test set

In [ ]:
# MLP (parameters)
# Differences from the previous lecture (lec04), all deliberate on Karpathy's part:
#   1. n_hidden: 200 -> 64
#   2. b1 is back (lec04 dropped it since BN cancels it out) — the official
#      notebook calls it "just for fun", but it's really a canary for the BN
#      backward pass: with a correct backward, db1 is exactly 0, so any bug in
#      the BN gradient immediately shows up as a nonzero db1.
#   3. Non-standard init (no more W2*0.01 / B2*0 / gain=1 / bias=0):
#      with lec04-style init, a wrong backward can cancel out "by accident"
#      and go undetected.

n_embd = 10    # dimension of embedding vectors
n_hidden = 64  # number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)  # X (xavier/kaiming init)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1  # "just for fun" (BN cancels it out; db1 == 0 if backward is correct)
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total

for p in parameters:
    p.requires_grad = True